# 01 — Certified spherical Bessel sequences: Sage export

This Binder-portable notebook evaluates all eight Mie boundary sequences
through order 42 with `ComplexBallField(192)` and exports midpoint pairs
for the two Go notebooks.

The Dockerfile applies the complete PR patch to the real Sage 10.9 source
tree before Jupyter starts. This notebook imports the three functions only
from `sage.all` and verifies that every function comes from the patched
`/home/sage/sage/src/sage/functions/bessel.py`. There is no fallback API.

Select the **SageMath** kernel and run all cells.


In [1]:
import hashlib
import inspect
import json
import sys
from pathlib import Path

import sage.functions.all as functions_all
from sage.all import ComplexBallField, QQ, RealBallField

PR_COMMIT = "dda09a7229a39589ef50dedcabd1c40de9d4f856"

from sage.all import (
    spherical_bessel_J_sequence,
    spherical_bessel_Y_sequence,
    spherical_hankel1_sequence,
)

API_MODE = "Sage 10.9 source patched with PR commit"

FUNCTIONS = (
    spherical_bessel_J_sequence,
    spherical_bessel_Y_sequence,
    spherical_hankel1_sequence,
)
API_SOURCE_FILES = [
    str(Path(inspect.getsourcefile(function)).resolve())
    for function in FUNCTIONS
]

expected_all = Path("/home/sage/sage/src/sage/functions/all.py").resolve()
expected_bessel = Path("/home/sage/sage/src/sage/functions/bessel.py").resolve()
assert Path(functions_all.__file__).resolve() == expected_all
assert all(Path(path) == expected_bessel for path in API_SOURCE_FILES)

print("Python:", Path(sys.executable).resolve())
print("API mode:", API_MODE)
print("sage.functions.all:", Path(functions_all.__file__).resolve())
for function, path in zip(FUNCTIONS, API_SOURCE_FILES):
    print(function.__name__, "->", path)
print("Sage PR commit:", PR_COMMIT)
print("API SOURCE CHECK: PASS")


Python: /usr/bin/python3.12
API mode: Sage 10.9 source patched with PR commit
sage.functions.all: /home/sage/sage/src/sage/functions/all.py
spherical_bessel_J_sequence -> /home/sage/sage/src/sage/functions/bessel.py
spherical_bessel_Y_sequence -> /home/sage/sage/src/sage/functions/bessel.py
spherical_hankel1_sequence -> /home/sage/sage/src/sage/functions/bessel.py
Sage PR commit: dda09a7229a39589ef50dedcabd1c40de9d4f856
API SOURCE CHECK: PASS


In [2]:
WIDTH = 800
HEIGHT = 800
MAX_ORDER = 42
WORKING_BITS = 192

CBF = ComplexBallField(WORKING_BITS)
RBF = RealBallField(WORKING_BITS)
radius_in_wavelengths = QQ(159) / 50
x = CBF(2) * CBF(RBF.pi()) * CBF(radius_in_wavelengths)
mx = CBF(QQ(133) / 100) * x

jX, jdX = spherical_bessel_J_sequence(MAX_ORDER, x)
jMX, jdMX = spherical_bessel_J_sequence(MAX_ORDER, mx)
yX, ydX = spherical_bessel_Y_sequence(MAX_ORDER, x)
hX, hdX = spherical_hankel1_sequence(MAX_ORDER, x)

sequences = {
    "JX": jX,
    "JXDerivative": jdX,
    "JMX": jMX,
    "JMXDerivative": jdMX,
    "YX": yX,
    "YXDerivative": ydX,
    "HX": hX,
    "HXDerivative": hdX,
}

assert all(len(values) == MAX_ORDER + 1 for values in sequences.values())
assert all(
    value.parent() is CBF
    for values in sequences.values()
    for value in values
)

imaginary_unit = CBF(0, 1)
assert all(
    (hX[ell] - jX[ell] - imaginary_unit * yX[ell]).contains_zero()
    for ell in range(MAX_ORDER + 1)
)
assert all(
    (hdX[ell] - jdX[ell] - imaginary_unit * ydX[ell]).contains_zero()
    for ell in range(MAX_ORDER + 1)
)

print("profile: full_800")
print("grid:", WIDTH, "x", HEIGHT)
print("8 sequences × 43 values; parent preservation: PASS")
print("H1 = J + iY containment: PASS")
print("X =", x)
print("MX =", mx)


profile: full_800
grid: 800 x 800
8 sequences × 43 values; parent preservation: PASS
H1 = J + iY containment: PASS
X = [19.9805292768310849966224119176576383434939973800256730214 +/- 1.21e-56]
MX = [26.5741039381853430455078078504846589968470165154341451185 +/- 7.15e-56]


In [3]:
def pair(ball):
    midpoint = ball.mid()
    return [float(midpoint.real()), float(midpoint.imag())]


output = {
    "Schema": "sage-spherical-sequences-for-go-v1",
    "MaxL": int(MAX_ORDER),
    "Width": int(WIDTH),
    "Height": int(HEIGHT),
    "WorkingBits": int(WORKING_BITS),
    "Lambda": float(QQ(1) / 5),
    "Radius": float(radius_in_wavelengths * QQ(1) / 5),
    "RefractiveIndex": [
        float(QQ(133) / 100),
        float(0),
    ],
    "Mu": float(1),
    "BoundaryArguments": {
        "X": pair(x),
        "MX": pair(mx),
    },
    "Sequences": {
        name: [pair(value) for value in values]
        for name, values in sequences.items()
    },
    "APIProvenance": {
        "Mode": API_MODE,
        "SagePRCommit": PR_COMMIT,
        "SourceFiles": API_SOURCE_FILES,
    },
}

output_path = Path.cwd() / "sage-spherical-sequences-for-go.json"
output_path.write_text(
    json.dumps(output, indent=int(2), sort_keys=True) + "\n",
    encoding="utf-8",
)
raw = output_path.read_bytes()

assert output_path.exists()
assert sum(
    len(values)
    for values in output["Sequences"].values()
) == 344
assert output["Width"] == output["Height"] == 800

print("wrote:", output_path.resolve())
print("SHA-256:", hashlib.sha256(raw).hexdigest())
print("JSON EXPORT: PASS (344 complex midpoint pairs; 800 x 800)")
print("NEXT: run 02_Go_Spherical_Sequence_Render.ipynb")

wrote: /home/jovyan/sage-spherical-sequences-for-go.json
SHA-256: ab8afa3f01b1b9bbbfe9c8ca938606ca7c74f2e3c2df99439a0e0ca5ab644fc7
JSON EXPORT: PASS (344 complex midpoint pairs; 800 x 800)
NEXT: run 02_Go_Spherical_Sequence_Render.ipynb
